# CARec encoder experiments

- Run shared setup and function cells once, then run only the model sections you choose.
- Each variant has one fixed folder containing only its two best model checkpoints. Rerunning replaces them.
- Metrics stay in notebook outputs and memory. Automatic resume is not implemented.


## Shared settings

- Model and MLP run on MPS; sparse graph propagation and uniformity run on CPU.
- Settings below apply to every model. Set both epoch limits to 1 for a short check.


In [1]:
from pathlib import Path
from copy import copy, deepcopy
from time import perf_counter
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.data.dataset import Dataset
from recbole.data.interaction import Interaction
from recbole.data.dataloader import FullSortEvalDataLoader
from recbole.sampler import Sampler
from recbole.evaluator import Collector, Evaluator
from recbole.utils import init_seed
from recbole.utils.enum_type import FeatureSource, FeatureType

carec = load_carec()

DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
EMBEDDING_DIR = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID
RUNS_DIR = PROJECT_ROOT / "artifacts/carec" / DATASET_ID

seed = 2023
phase1_epochs = 90
phase2_epochs = 30
patience = 15
weight_decay = 0.0
training_batch_size = 2048
learning_rate = 0.001

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable.")
device = torch.device("mps")


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Shared data

- Load interactions and RecBole ID mappings once. Keep the saved splits and repeated interactions.
- Padding has ID 0. Only training edges enter the model graph.


In [2]:
RECBOLE_DIR.mkdir(parents=True, exist_ok=True)
interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]
for split_name in ("train", "valid", "test", "cold"):
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    if not split_path.exists():
        interactions = pd.read_csv(
            PREPARED_DIR / f"{split_name}.tsv.gz",
            sep="\t",
            usecols=interaction_columns,
            dtype={"user_id:token": str, "item_id:token": str},
            float_precision="round_trip",
        )
        # RecBole requires typed headers. Leave source_row out of this export.
        interactions[interaction_columns].to_csv(split_path, sep="\t", index=False)

catalogue_path = RECBOLE_DIR / f"{DATASET_ID}.item"
if not catalogue_path.exists():
    catalogue_ids = pd.read_csv(PREPARED_DIR / "item_ids.tsv.gz", sep="\t", dtype={"item_id": str})
    pd.DataFrame({"item_id:token": catalogue_ids["item_id"]}).to_csv(
        catalogue_path, sep="\t", index=False,
    )



In [3]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,
}

base_config = Config(model=carec.CollabContex, dataset=DATASET_ID, config_dict=dataset_settings)
# Set before construction: CARec stores graph and item tensors outside registered buffers.
base_config["device"] = device
recbole_dataset = Dataset(base_config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]
train_dataset, valid_dataset, test_dataset, cold_dataset = recbole_dataset.build()

base_config["metrics"] = ["Recall", "NDCG"]
base_config["topk"] = [10, 50]
base_config["eval_batch_size"] = 64 * train_dataset.item_num


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on

In [4]:
# Upstream repeatable sampling leaves history empty. Supply cumulative history.
history_sampler = Sampler(
    ["train", "valid", "test", "cold"],
    [train_dataset, valid_dataset, test_dataset, cold_dataset],
)
evaluation_loaders = {}
for split_name, split in [("valid", valid_dataset), ("warm", test_dataset), ("cold", cold_dataset)]:
    phase = "test" if split_name == "warm" else split_name
    evaluation_loaders[split_name] = FullSortEvalDataLoader(
        base_config, split, history_sampler.set_phase(phase), shuffle=False,
    )

cold_product_ids = cold_dataset.inter_feat[base_config["ITEM_ID_FIELD"]].unique()
train_product_ids = train_dataset.inter_feat[base_config["ITEM_ID_FIELD"]].unique()
assert not set(cold_product_ids.tolist()) & set(train_product_ids.tolist())
print({name: len(loader.uid_list) for name, loader in evaluation_loaders.items()})


{'valid': 12386, 'warm': 81520, 'cold': 18680}


## Prepare one model

- Match cached vectors to RecBole IDs and add zero padding. Each variant gets separate item features and a fresh model.
- Keep original normalized edge weights. CPU sparse multiplication replaces edge-wise messages with differentiable transfers to MPS.
- Native dimensions also change user-table and MLP sizes. This is not yet a comparison at a common dimension.


In [5]:
def prepare_model(embedding_name):
    embedding_path = EMBEDDING_DIR / embedding_name
    with np.load(embedding_path, allow_pickle=False) as saved:
        item_ids = saved["item_ids"]
        vectors = saved["embeddings"]
        metadata = {
            "embedding_file": str(embedding_path.relative_to(PROJECT_ROOT)),
            "encoder": str(saved["model_id"]), "revision": str(saved["revision"]),
            "prompt": str(saved["prompt"]), "dimension": vectors.shape[1],
        }
    rows = {product_id: row for row, product_id in enumerate(item_ids)}
    assert len(rows) == len(item_ids), "Duplicate cache IDs"
    assert len(vectors) == len(item_ids), "Cache row count differs"
    assert recbole_product_ids[0] == "[PAD]"
    assert set(recbole_product_ids[1:]) == set(item_ids), "Product catalogues differ"
    aligned = np.zeros((train_dataset.item_num, vectors.shape[1]), dtype=np.float32)
    for item_id, product_id in enumerate(recbole_product_ids[1:], start=1):
        aligned[item_id] = vectors[rows[product_id]]

    config = deepcopy(base_config)
    dimension = metadata["dimension"]
    config["seed"] = seed
    config["encoder"] = "LightGCN"
    config["num_layers"] = 2
    config["embedding_size"] = dimension
    config["mlp"] = {
        "input_dim": dimension, "hidden_dims": [dimension, dimension],
        "output_dim": dimension, "num_layers": 2, "dropout": 0.2,
    }
    # Share immutable interactions; give this variant its own item features and metadata.
    model_dataset = copy(train_dataset)
    model_dataset.config = config
    for field in ("field2type", "field2source", "field2seqlen"):
        setattr(model_dataset, field, getattr(train_dataset, field).copy())
    model_dataset.set_field_property("item_emb", FeatureType.FLOAT_SEQ, FeatureSource.ITEM, dimension)
    model_dataset.item_feat = Interaction({
        **train_dataset.item_feat.interaction, "item_emb": torch.from_numpy(aligned),
    })
    init_seed(seed, config["reproducibility"])
    model = carec.CollabContex(config, model_dataset).to(device)
    encoder = model.encoder
    node_count = encoder.n_users + encoder.n_items
    adjacency = torch.sparse_coo_tensor(
        encoder.edge_index.flip(0).cpu(), encoder.edge_weight.cpu(), (node_count, node_count),
    ).coalesce().to_sparse_csr()

    def sparse_forward(node_vectors, edge_index, edge_weight):
        return torch.sparse.mm(adjacency, node_vectors.cpu()).to(node_vectors.device)

    def cpu_uniformity(vectors, t=2):
        return carec.CollabContex.uniformity(vectors.cpu(), t=t).to(vectors.device)

    encoder.gcn_conv.forward = sparse_forward
    model.uniformity = cpu_uniformity
    return model, config, metadata


## Evaluation

- RecBole Recall/NDCG at 10 and 50, averaged over users with positives. Warm ranks the full catalogue; cold ranks cold products with MLP bypassed.
- Mask padding and earlier history, except current positives. Repeated pairs across splits remain a benchmark limitation.


In [6]:
@torch.no_grad()
def evaluate_carec(model, loader, config, cold_only=False):
    collector = Collector(config)
    item_count = loader.dataset.item_num
    evaluator = Evaluator(config)
    previous_training = model.training
    previous_mlp_bypass = model.encoder.not_apply_mlp
    model.eval()
    if cold_only:
        model.encoder.not_apply_mlp = True
    # full_sort_predict caches graph outputs. Never reuse them across modes or epochs.
    model.restore_user_e = model.restore_item_e = None
    candidate_mask = torch.ones(item_count, dtype=torch.bool, device=config["device"])
    if cold_only:
        candidate_mask[:] = False
        cold_ids = loader.dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
        candidate_mask[cold_ids.to(config["device"])] = True
    candidate_mask[0] = False

    try:
        for interaction, history, positive_users, positive_items in tqdm(loader, desc="Cold" if cold_only else "Full ranking"):
            scores = model.full_sort_predict(interaction.to(config["device"]))
            scores = scores.view(-1, item_count)
            if not torch.isfinite(scores).all():
                raise ValueError("Non-finite recommendation scores before masking.")
            # Apply the item mask to columns, for every user in the batch.
            scores[:, ~candidate_mask] = -torch.inf
            if history is not None:
                scores[history[0].to(scores.device), history[1].to(scores.device)] = -torch.inf
            collector.eval_batch_collect(scores, interaction, positive_users, positive_items)
        return {name: float(value) for name, value in evaluator.evaluate(collector.get_data_struct()).items()}
    finally:
        model.encoder.not_apply_mlp = previous_mlp_bypass
        model.restore_user_e = model.restore_item_e = None
        model.train(previous_training)


## Saved results

- Save each completed evaluation under `results/carec/<dataset>/<variant>/`, separately as `carec.json` and `content-only.json`. New evaluations replace the matching result through a temporary file.
- Keep model checkpoints unchanged. Result files contain metrics and available run details; checkpoint reuse cannot recover historical training settings.
- The final table loads saved results, including after a kernel restart. It also writes `summary.csv` without timing or epoch columns.


In [21]:
import json
from datetime import datetime, timezone

RESULTS_DIR = PROJECT_ROOT / "results/carec" / DATASET_ID
RESULT_FILES = {"CARec": "carec.json", "Content-only": "content-only.json"}


def save_experiment_result(embedding_name, method, result):
    variant = Path(embedding_name).stem
    result_path = RESULTS_DIR / variant / RESULT_FILES[method]
    result_path.parent.mkdir(parents=True, exist_ok=True)
    record = {
        "schema_version": 1,
        "dataset_id": DATASET_ID,
        "embedding_file": Path(embedding_name).name,
        "method": method,
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "result": result,
    }
    temporary = result_path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(record, indent=2, allow_nan=False) + "\n")
    temporary.replace(result_path)


def load_experiment_results(method):
    results = {}
    for result_path in sorted(RESULTS_DIR.glob(f"*/{RESULT_FILES[method]}")):
        record = json.loads(result_path.read_text())
        variant = result_path.parent.name
        if (record["schema_version"] != 1 or record["dataset_id"] != DATASET_ID
                or record["method"] != method
                or Path(record["embedding_file"]).stem != variant):
            raise ValueError(f"Result identity differs: {result_path}")
        results[variant] = record["result"]
    return results


## Content-only baseline

- Average cached item vectors over each user's training interactions, including repeats. Rank by cosine similarity, without CARec training or checkpoints.
- Reuse CARec's evaluation users, warm/cold candidates, history masks, and Recall/NDCG metrics. Run on CPU.
- Empty or zero profiles score all candidates equally. Keep these users and report their counts; ties use the same RecBole top-k routine as CARec.
- Each model cell calls this baseline on the line after `run_experiment`. Results stay in `content_only_results` and are saved to JSON; the final table shows both methods.
- For baseline-only evaluation, call `minilm_content_result = evaluate_content_only("all-MiniLM-L6-v2.npz")`.


In [7]:
content_only_results = globals().get("content_only_results", {})


@torch.no_grad()
def evaluate_content_only(embedding_name):
    started = perf_counter()
    config = deepcopy(base_config)
    config["device"] = torch.device("cpu")
    with np.load(EMBEDDING_DIR / embedding_name, allow_pickle=False) as saved:
        item_ids = saved["item_ids"]
        vectors = saved["embeddings"]
    if vectors.ndim != 2 or len(vectors) != len(item_ids):
        raise ValueError("Cache must contain one vector per item ID.")
    rows = {product_id: row for row, product_id in enumerate(item_ids)}
    if len(rows) != len(item_ids):
        raise ValueError("Duplicate cache IDs.")
    if recbole_product_ids[0] != "[PAD]" or set(recbole_product_ids[1:]) != set(item_ids):
        raise ValueError("Cache and RecBole product catalogues differ.")
    aligned = np.zeros((train_dataset.item_num, vectors.shape[1]), dtype=np.float32)
    for item_id, product_id in enumerate(recbole_product_ids[1:], start=1):
        aligned[item_id] = vectors[rows[product_id]]
    if not np.isfinite(aligned).all():
        raise ValueError("Non-finite item embeddings.")
    item_vectors = torch.from_numpy(aligned)
    if (item_vectors[1:].norm(dim=1) == 0).any():
        raise ValueError("An item has a zero embedding.")

    # Preserve repeated training interactions as repeated contributions to the mean.
    history = train_dataset.inter_matrix(form="csr").astype(np.float32)
    counts = np.asarray(history.sum(axis=1)).ravel()
    mean_history = history.multiply((1.0 / np.maximum(counts, 1))[:, None]).tocsr()
    profiles = torch.from_numpy(np.asarray(mean_history @ aligned, dtype=np.float32))
    profile_norms = profiles.norm(dim=1)
    user_vectors = torch.nn.functional.normalize(profiles, dim=1)
    item_vectors = torch.nn.functional.normalize(item_vectors, dim=1)
    profile_seconds = perf_counter() - started
    del vectors, aligned, history, mean_history, profiles

    test = {}
    zero_profile_users = {}
    scoring_started = perf_counter()
    for split_name in ("warm", "cold"):
        loader = evaluation_loaders[split_name]
        collector = Collector(config)
        evaluator = Evaluator(config)
        candidate_mask = torch.ones(train_dataset.item_num, dtype=torch.bool)
        if split_name == "cold":
            candidate_mask[:] = False
            cold_ids = loader.dataset.inter_feat[config["ITEM_ID_FIELD"]].unique().cpu()
            candidate_mask[cold_ids] = True
        candidate_mask[0] = False
        zero_profile_users[split_name] = 0
        for interaction, history, positive_users, positive_items in tqdm(loader, desc=f"Content-only {split_name}"):
            user_ids = interaction[config["USER_ID_FIELD"]].cpu()
            zero_profile_users[split_name] += int((profile_norms[user_ids] == 0).sum())
            scores = user_vectors[user_ids] @ item_vectors.T
            if not torch.isfinite(scores).all():
                raise ValueError("Non-finite content-only scores before masking.")
            scores[:, ~candidate_mask] = -torch.inf
            if history is not None:
                scores[history[0].cpu(), history[1].cpu()] = -torch.inf
            collector.eval_batch_collect(scores, interaction, positive_users, positive_items)
        test[split_name] = {
            name: float(value)
            for name, value in evaluator.evaluate(collector.get_data_struct()).items()
        }
    result = {
        "dimension": item_vectors.shape[1], "test": test,
        "zero_profile_users": zero_profile_users,
        "profile_seconds": profile_seconds,
        "scoring_seconds": perf_counter() - scoring_started,
        "total_seconds": perf_counter() - started,
    }
    content_only_results[Path(embedding_name).stem] = result
    save_experiment_result(embedding_name, "Content-only", result)
    display(pd.DataFrame(test).T)
    print(f"Zero-profile users: {zero_profile_users}")
    return result


## Train and save

- Phase 1 learns user vectors. Phase 2 freezes them and trains MLP, starting from the best phase-one weights.
- Validate every epoch; select by NDCG@10 and stop at the configured patience. Evaluate test splits only after training.
- Save only `phase1-best.pt` and `phase2-best.pt`, containing model weights. Rebuilding the model requires the same embedding cache and dataset ID mapping.


In [8]:
def train_phase(model, config, phase, max_epochs, run_dir):
    if max_epochs < 1 or patience < 1:
        raise ValueError("Epoch limit and patience must be positive.")
    if phase == 1:
        model.set_item_tutoring_phase()
        uniformity_field = config["USER_ID_FIELD"]
    elif phase == 2:
        model.set_user_tutoring_phase()
        uniformity_field = config["ITEM_ID_FIELD"]
    else:
        raise ValueError("CARec has phases 1 and 2.")

    # Frozen parameters have no optimizer state in this phase.
    model.zero_grad(set_to_none=True)
    optimizer = torch.optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate, weight_decay=weight_decay,
    )
    generator = torch.Generator().manual_seed(seed + phase)
    checkpoint_path = run_dir / f"phase{phase}-best.pt"
    best_score, stale_epochs = -float("inf"), 0
    best_epoch, best_metrics = None, None
    history = []
    started = perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        indices = torch.randperm(len(train_dataset.inter_feat), generator=generator)
        total_loss, interaction_count = 0.0, 0
        for batch_ids in tqdm(indices.split(training_batch_size), desc=f"Phase {phase}, epoch {epoch}"):
            batch = train_dataset.inter_feat[batch_ids].to(config["device"])
            if batch[uniformity_field].unique().numel() < 2:
                raise ValueError("Uniformity needs at least two distinct active-side IDs.")
            optimizer.zero_grad(set_to_none=True)
            loss = model.calculate_loss(batch)
            if not torch.isfinite(loss):
                raise ValueError("Training loss is not finite.")
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
            interaction_count += len(batch)

        valid_metrics = evaluate_carec(model, evaluation_loaders["valid"], config)
        score = float(valid_metrics["ndcg@10"])
        if not np.isfinite(score):
            raise ValueError("Validation NDCG@10 is not finite.")
        improved = score > best_score
        stale_epochs = 0 if improved else stale_epochs + 1
        history.append({
            "epoch": epoch, "loss": total_loss / interaction_count,
            "valid": valid_metrics, "elapsed_seconds": perf_counter() - started,
        })
        if improved:
            best_score = score
            # Write a temporary file so interruption cannot truncate the previous best.
            temporary = checkpoint_path.with_suffix(".pt.tmp")
            best_epoch, best_metrics = epoch, valid_metrics
            torch.save(
                {name: value.detach().cpu() for name, value in model.state_dict().items()},
                temporary,
            )
            temporary.replace(checkpoint_path)
        print(f"Phase {phase}, epoch {epoch}: loss={history[-1]['loss']:.4f}, valid NDCG@10={score:.4f}, patience={stale_epochs}/{patience}")
        if stale_epochs >= patience:
            break

    # Reload the best weights before the next phase or final evaluation.
    best = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(best)
    model.restore_user_e = model.restore_item_e = None
    model.eval()
    return {"best_epoch": best_epoch, "valid": best_metrics, "epochs_run": len(history)}


In [9]:
experiment_results = globals().get("experiment_results", {})


def run_experiment(embedding_name, *, phase2_only=False):
    variant = Path(embedding_name).stem
    run_dir = RUNS_DIR / variant
    checkpoints = sorted(path for path in run_dir.glob("*.pt") if path.is_file())
    if phase2_only and not (run_dir / "phase1-best.pt").is_file():
        raise FileNotFoundError(f"Phase 2 requires {run_dir / 'phase1-best.pt'}")
    started = perf_counter()
    model, config, metadata = prepare_model(embedding_name)
    if phase2_only and not (run_dir / "phase2-best.pt").is_file():
        checkpoint_path = run_dir / "phase1-best.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
        model.restore_user_e = model.restore_item_e = None
        print(f"Phase 2 only: {embedding_name}, loaded {checkpoint_path.name}")
        training = {"phase1": {"best_epoch": None}}
        training["phase2"] = train_phase(model, config, 2, phase2_epochs, run_dir)
    elif checkpoints:
        checkpoint_path = next(
            (run_dir / name for name in ("phase2-best.pt", "phase1-best.pt")
             if (run_dir / name).is_file()),
            None,
        )
        if checkpoint_path is None:
            raise ValueError(f"Checkpoint exists in {run_dir}, but no recognized phase checkpoint was found. Training skipped.")
        model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
        if checkpoint_path.name == "phase2-best.pt":
            model.set_user_tutoring_phase()
        else:
            model.set_item_tutoring_phase()
        model.restore_user_e = model.restore_item_e = None
        model.eval()
        print(f"Evaluation only: {embedding_name}, loaded {checkpoint_path.name}")
        training = {"phase1": {"best_epoch": None}, "phase2": {"best_epoch": None}}
    else:
        run_dir.mkdir(parents=True, exist_ok=True)
        experiment_results.pop(variant, None)
        print(f"{embedding_name}: {metadata['dimension']} dimensions, model {device}, graph CPU")
        print(f"Model directory: {run_dir}")
        training = {}
        training["phase1"] = train_phase(model, config, 1, phase1_epochs, run_dir)
        training["phase2"] = train_phase(model, config, 2, phase2_epochs, run_dir)
    test = {
        "warm": evaluate_carec(model, evaluation_loaders["warm"], config),
        "cold": evaluate_carec(model, evaluation_loaders["cold"], config, cold_only=True),
    }
    result = {
        "dimension": metadata["dimension"], "training": training, "test": test,
        "total_seconds": perf_counter() - started, "run_dir": str(run_dir),
    }
    experiment_results[variant] = result
    save_experiment_result(embedding_name, "CARec", result)
    print("CARec")
    display(pd.DataFrame(test).T)
    return result


## Individual runs

- Run each cell independently after the shared cells above. Definitions alone do not start training.
- Returned values contain metrics and paths, not the trained model.


### MiniLM-L6-v2

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [10]:
minilm_result = run_experiment("all-MiniLM-L6-v2.npz")
minilm_content_result = evaluate_content_only("all-MiniLM-L6-v2.npz")


/var/folders/wz/x7xqj0yj6ls_367syhygxjn40000gn/T/ipykernel_65941/822926853.py:45: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  ).coalesce().to_sparse_csr()


Evaluation only: all-MiniLM-L6-v2.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 117.75it/s]


CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0714,0.1050,0.0619,0.0702
cold,0.0532,0.1344,0.0304,0.0488


Content-only cold: 100%|██████████| 292/292 [00:01<00:00, 174.24it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0269,0.0489,0.0194,0.0247
cold,0.0479,0.1222,0.0290,0.0454


Zero-profile users: {'warm': 0, 'cold': 0}


### MPNet-base-v2

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [11]:
mpnet_result = run_experiment("all-mpnet-base-v2.npz")
mpnet_content_result = evaluate_content_only("all-mpnet-base-v2.npz")


Evaluation only: all-mpnet-base-v2.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 99.27it/s] 

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0718,0.1046,0.0626,0.0708
cold,0.0551,0.1427,0.0326,0.0524


Content-only cold: 100%|██████████| 292/292 [00:01<00:00, 147.52it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0258,0.0456,0.0183,0.0232
cold,0.0502,0.1253,0.0300,0.0470


Zero-profile users: {'warm': 0, 'cold': 0}


### E5-base-v2

- Input: without task instruction, query prefix. The exact prompt is read from the cache and saved with the run.


In [12]:
e5_base_result = run_experiment("e5-base-v2.npz")
e5_base_content_result = evaluate_content_only("e5-base-v2.npz")


Evaluation only: e5-base-v2.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 105.81it/s]

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0721,0.1051,0.0630,0.0711
cold,0.0530,0.1364,0.0305,0.0493


Content-only cold: 100%|██████████| 292/292 [00:01<00:00, 147.70it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0359,0.0576,0.0275,0.0328
cold,0.0532,0.1312,0.0332,0.0506


Zero-profile users: {'warm': 0, 'cold': 0}


### Qwen3-Embedding-0.6B

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [13]:
qwen_plain_result = run_experiment("Qwen3-Embedding-0.6B-without_instruction.npz", phase2_only=True)
qwen_plain_content_result = evaluate_content_only("Qwen3-Embedding-0.6B-without_instruction.npz")


Evaluation only: Qwen3-Embedding-0.6B-without_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:03<00:00, 91.89it/s] 

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0724,0.1069,0.0627,0.0712
cold,0.0588,0.1431,0.0352,0.0542


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 129.97it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0199,0.0373,0.0142,0.0184
cold,0.0427,0.1015,0.0259,0.0391


Zero-profile users: {'warm': 0, 'cold': 0}


### Qwen3-Embedding-0.6B

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [14]:
qwen_instruct_result = run_experiment("Qwen3-Embedding-0.6B-with_instruction.npz")
qwen_instruct_content_result = evaluate_content_only("Qwen3-Embedding-0.6B-with_instruction.npz")


Evaluation only: Qwen3-Embedding-0.6B-with_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 97.59it/s] 


CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0724,0.1050,0.063,0.0711
cold,0.0534,0.1367,0.031,0.0497


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 131.02it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0199,0.0382,0.0143,0.0187
cold,0.0455,0.1158,0.0264,0.0421


Zero-profile users: {'warm': 0, 'cold': 0}


### Multilingual E5-large-instruct

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [15]:
e5_large_plain_result = run_experiment("multilingual-e5-large-instruct-without_instruction.npz")
e5_large_plain_content_result = evaluate_content_only("multilingual-e5-large-instruct-without_instruction.npz")


Evaluation only: multilingual-e5-large-instruct-without_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 101.20it/s]


CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0723,0.1053,0.0631,0.0713
cold,0.0555,0.1405,0.0321,0.0514


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 120.43it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0325,0.0543,0.0247,0.0300
cold,0.0530,0.1267,0.0329,0.0495


Zero-profile users: {'warm': 0, 'cold': 0}


### Multilingual E5-large-instruct

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [16]:
e5_large_instruct_result = run_experiment("multilingual-e5-large-instruct-with_instruction.npz",phase2_only=True)
e5_large_instruct_content_result = evaluate_content_only("multilingual-e5-large-instruct-with_instruction.npz")


Evaluation only: multilingual-e5-large-instruct-with_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 97.60it/s] 

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0719,0.1050,0.063,0.0712
cold,0.0506,0.1353,0.029,0.0482


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 136.28it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0267,0.0466,0.0203,0.0251
cold,0.0485,0.1202,0.0294,0.0455


Zero-profile users: {'warm': 0, 'cold': 0}


### INSTRUCTOR-base

- Input: without task instruction. The exact prompt is read from the cache and saved with the run.


In [17]:
instructor_plain_result = run_experiment("instructor-base-without_instruction.npz")
instructor_plain_content_result = evaluate_content_only("instructor-base-without_instruction.npz")



Evaluation only: instructor-base-without_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 107.12it/s]

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0719,0.1055,0.0628,0.0711
cold,0.0516,0.1372,0.0287,0.0479


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 145.06it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0239,0.0430,0.0172,0.0218
cold,0.0441,0.1128,0.0258,0.0410


Zero-profile users: {'warm': 0, 'cold': 0}


### INSTRUCTOR-base

- Input: with task instruction. The exact prompt is read from the cache and saved with the run.


In [18]:
instructor_instruct_result = run_experiment("instructor-base-with_instruction.npz")
instructor_instruct_content_result = evaluate_content_only("instructor-base-with_instruction.npz")


Evaluation only: instructor-base-with_instruction.npz, loaded phase2-best.pt


Cold: 100%|██████████| 292/292 [00:02<00:00, 105.02it/s]

CARec


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0721,0.1048,0.0631,0.0712
cold,0.0470,0.1316,0.0265,0.0455


Content-only cold: 100%|██████████| 292/292 [00:02<00:00, 144.05it/s]


,recall@10,recall@50,ndcg@10,ndcg@50
warm,0.0229,0.0422,0.0164,0.0211
cold,0.0432,0.1134,0.0256,0.0413


Zero-profile users: {'warm': 0, 'cold': 0}


## Results

- Show CARec and content-only metrics from this session and saved JSON files. Each method can appear independently.
- Save the table as `results/carec/<dataset>/summary.csv`. Show variant, method, dimension, and warm/cold metrics only.
- After a kernel restart, run shared setup, the saved-results definitions, and this cell to display saved results without evaluation or training.


In [22]:
saved_results = {method: load_experiment_results(method) for method in RESULT_FILES}
variants = sorted({variant for results in saved_results.values() for variant in results})
result_rows = []
for variant in variants:
    for method, results in saved_results.items():
        if variant not in results:
            continue
        result = results[variant]
        row = {"variant": variant, "method": method, "dimension": result["dimension"]}
        for split_name, metrics in result["test"].items():
            row.update({f"{split_name}_{metric}": value for metric, value in metrics.items()})
        result_rows.append(row)
summary = pd.DataFrame(result_rows)
if not summary.empty:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    temporary = RESULTS_DIR / "summary.csv.tmp"
    summary.to_csv(temporary, index=False)
    temporary.replace(RESULTS_DIR / "summary.csv")
display(summary)


,variant,method,dimension,warm_recall@10,warm_recall@50,warm_ndcg@10,warm_ndcg@50,cold_recall@10,cold_recall@50,cold_ndcg@10,cold_ndcg@50
0,Qwen3-Embedding-0.6B-with_instruction,CARec,1024,0.0724,0.1050,0.0630,0.0711,0.0534,0.1367,0.0310,0.0497
1,Qwen3-Embedding-0.6B-with_instruction,Content-only,1024,0.0199,0.0382,0.0143,0.0187,0.0455,0.1158,0.0264,0.0421
2,Qwen3-Embedding-0.6B-without_instruction,CARec,1024,0.0724,0.1069,0.0627,0.0712,0.0588,0.1431,0.0352,0.0542
3,Qwen3-Embedding-0.6B-without_instruction,Content-only,1024,0.0199,0.0373,0.0142,0.0184,0.0427,0.1015,0.0259,0.0391
4,all-MiniLM-L6-v2,CARec,384,0.0714,0.1050,0.0619,0.0702,0.0532,0.1344,0.0304,0.0488
5,all-MiniLM-L6-v2,Content-only,384,0.0269,0.0489,0.0194,0.0247,0.0479,0.1222,0.0290,0.0454
6,all-mpnet-base-v2,CARec,768,0.0718,0.1046,0.0626,0.0708,0.0551,0.1427,0.0326,0.0524
7,all-mpnet-base-v2,Content-only,768,0.0258,0.0456,0.0183,0.0232,0.0502,0.1253,0.0300,0.0470
8,e5-base-v2,CARec,768,0.0721,0.1051,0.0630,0.0711,0.0530,0.1364,0.0305,0.0493
9,e5-base-v2,Content-only,768,0.0359,0.0576,0.0275,0.0328,0.0532,0.1312,0.0332,0.0506
